# TPU-Native "Pallas" FlashAttention (Kaggle TPU v3-8)

This notebook hand-writes a **FlashAttention** kernel in **JAX Pallas** — the
closest thing to "writing CUDA" for Google's TPUs — and runs it inside a tiny
LLaMA-style decoder.

Instead of letting the XLA compiler fuse a naive `softmax(Q Kᵀ) V` (which
materializes the full `[seq, seq]` score matrix in HBM), we:

1. **Tile** Q/K/V and map each block from **HBM → VMEM** explicitly via Pallas
   `BlockSpec` index maps.
2. Run the **online-softmax** recurrence so the score matrix is never fully
   materialized.
3. Skip causal blocks that lie entirely in the future.

> **Setup:** In the Kaggle sidebar set **Accelerator → TPU VM v3-8**, then run
> all cells top to bottom.

## 1. Setup JAX & confirm the TPU

In [ ]:
# On Kaggle TPU VMs JAX is usually preinstalled. If not, uncomment:
# !pip install -q "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
import jax, jax.numpy as jnp
print("JAX", jax.__version__)
print("Devices:", jax.devices())
assert any(d.platform == "tpu" for d in jax.devices()), "No TPU found — set Accelerator to TPU v3-8"


## 2. The Pallas FlashAttention kernel

The `index_map` lambdas below are the **manual HBM → VMEM mapping**: for each
grid point `(batch, head, q_block, kv_block)` they say exactly which block of the
big HBM array to stream into the chip's small VMEM scratchpad.

In [ ]:
import functools
from jax.experimental import pallas as pl
from jax.experimental.pallas import tpu as pltpu

_NEG_INF = -1e30

def _flash_attention_kernel(q_ref, k_ref, v_ref, o_ref,
                            m_scratch, l_scratch, acc_scratch,
                            *, sm_scale, causal, block_q, block_k,
                            seq_len_q, kv_len):
    q_block_idx = pl.program_id(2)
    kv_block_idx = pl.program_id(3)
    num_kv_blocks = pl.num_programs(3)

    @pl.when(kv_block_idx == 0)
    def _init():
        m_scratch[...] = jnp.full_like(m_scratch, _NEG_INF)
        l_scratch[...] = jnp.zeros_like(l_scratch)
        acc_scratch[...] = jnp.zeros_like(acc_scratch)

    def _do_block():
        q = q_ref[0, 0].astype(jnp.float32)
        k = k_ref[0, 0].astype(jnp.float32)
        v = v_ref[0, 0].astype(jnp.float32)

        kv_idx = kv_block_idx * block_k + jax.lax.broadcasted_iota(jnp.int32, (block_k, 1), 0)
        kv_valid = kv_idx < kv_len
        k = jnp.where(kv_valid, k, 0.0)
        v = jnp.where(kv_valid, v, 0.0)

        s = jnp.dot(q, k.T, preferred_element_type=jnp.float32) * sm_scale
        q_pos = q_block_idx * block_q + jax.lax.broadcasted_iota(jnp.int32, (block_q, block_k), 0)
        k_pos = kv_block_idx * block_k + jax.lax.broadcasted_iota(jnp.int32, (block_q, block_k), 1)
        mask = k_pos < kv_len
        if causal:
            mask = jnp.logical_and(mask, q_pos >= k_pos)
        s = jnp.where(mask, s, _NEG_INF)

        m_prev = m_scratch[...]
        m_cur = jnp.max(s, axis=-1, keepdims=True)
        m_new = jnp.maximum(m_prev, m_cur)
        p = jnp.exp(s - m_new)
        alpha = jnp.exp(m_prev - m_new)
        l_scratch[...] = alpha * l_scratch[...] + jnp.sum(p, axis=-1, keepdims=True)
        acc_scratch[...] = acc_scratch[...] * alpha + jnp.dot(p, v, preferred_element_type=jnp.float32)
        m_scratch[...] = m_new

    # Skip blocks that are entirely masked: past the cache length, or (causal)
    # entirely in the future of this query block.
    first_k_pos = kv_block_idx * block_k
    live = first_k_pos < kv_len
    if causal:
        live = jnp.logical_and(live, q_block_idx * block_q + (block_q - 1) >= first_k_pos)
    pl.when(live)(_do_block)

    @pl.when(kv_block_idx == num_kv_blocks - 1)
    def _finalize():
        l = jnp.where(l_scratch[...] == 0.0, 1.0, l_scratch[...])
        o_ref[0, 0] = (acc_scratch[...] / l).astype(o_ref.dtype)


def flash_attention(q, k, v, *, causal=False, sm_scale=None, kv_len=None,
                    block_q=128, block_k=128, interpret=False):
    batch, num_heads, seq_len_q, head_dim = q.shape
    num_kv_heads, seq_len_k = k.shape[1], k.shape[2]
    q_per_kv = num_heads // num_kv_heads   # 1 for MHA, >1 for grouped-query attn
    if sm_scale is None:
        sm_scale = 1.0 / (head_dim ** 0.5)
    if kv_len is None:
        kv_len = seq_len_k   # pass a smaller value for a partially-filled cache
    grid = (batch, num_heads, pl.cdiv(seq_len_q, block_q), pl.cdiv(seq_len_k, block_k))
    q_spec = pl.BlockSpec((1, 1, block_q, head_dim), lambda b, h, i, j: (b, h, i, 0))
    # Each query head h reads KV head h // q_per_kv (grouped-query attention).
    k_spec = pl.BlockSpec((1, 1, block_k, head_dim), lambda b, h, i, j: (b, h // q_per_kv, j, 0))
    o_spec = pl.BlockSpec((1, 1, block_q, head_dim), lambda b, h, i, j: (b, h, i, 0))
    kernel = functools.partial(_flash_attention_kernel, sm_scale=sm_scale, causal=causal,
                               block_q=block_q, block_k=block_k,
                               seq_len_q=seq_len_q, kv_len=kv_len)
    return pl.pallas_call(
        kernel, grid=grid, in_specs=[q_spec, k_spec, k_spec], out_specs=o_spec,
        out_shape=jax.ShapeDtypeStruct(q.shape, q.dtype),
        scratch_shapes=[pltpu.VMEM((block_q, 1), jnp.float32),
                        pltpu.VMEM((block_q, 1), jnp.float32),
                        pltpu.VMEM((block_q, head_dim), jnp.float32)],
        compiler_params=pltpu.CompilerParams(
            dimension_semantics=("parallel", "parallel", "parallel", "arbitrary")),
        interpret=interpret, name="flash_attention_fwd")(q, k, v)


## 3. Correctness check vs. a naive reference attention

In [ ]:
def reference_attention(q, k, v, *, causal=False, sm_scale=None):
    head_dim = q.shape[-1]
    if sm_scale is None:
        sm_scale = 1.0 / (head_dim ** 0.5)
    scores = jnp.einsum("bhqd,bhkd->bhqk", q, k).astype(jnp.float32) * sm_scale
    if causal:
        sq, sk = q.shape[2], k.shape[2]
        qp = jax.lax.broadcasted_iota(jnp.int32, (sq, sk), 0)
        kp = jax.lax.broadcasted_iota(jnp.int32, (sq, sk), 1)
        scores = jnp.where(qp >= kp, scores, -1e30)
    w = jax.nn.softmax(scores, axis=-1)
    return jnp.einsum("bhqk,bhkd->bhqd", w, v).astype(q.dtype)

keys = jax.random.split(jax.random.PRNGKey(0), 3)
q = jax.random.normal(keys[0], (1, 4, 512, 128), jnp.float32)
k = jax.random.normal(keys[1], (1, 4, 512, 128), jnp.float32)
v = jax.random.normal(keys[2], (1, 4, 512, 128), jnp.float32)
for causal in (False, True):
    out = flash_attention(q, k, v, causal=causal)
    ref = reference_attention(q, k, v, causal=causal)
    print(f"causal={causal}: max abs error = {float(jnp.max(jnp.abs(out - ref))):.2e}")


## 4. Benchmark: custom Pallas kernel vs. default XLA attention

In [ ]:
import time

def bench(fn, *args, warmup=3, iters=20):
    for _ in range(warmup):
        jax.block_until_ready(fn(*args))
    t0 = time.perf_counter()
    for _ in range(iters):
        out = fn(*args)
    jax.block_until_ready(out)
    return (time.perf_counter() - t0) / iters

pallas_fn = jax.jit(lambda q, k, v: flash_attention(q, k, v, causal=True))
xla_fn = jax.jit(lambda q, k, v: reference_attention(q, k, v, causal=True))

print(f"{'seq_len':>8} | {'XLA (ms)':>10} | {'Pallas (ms)':>12} | {'speedup':>8}")
print("-" * 48)
for seq in [256, 512, 1024, 2048, 4096]:
    ks = jax.random.split(jax.random.PRNGKey(seq), 3)
    q = jax.random.normal(ks[0], (1, 8, seq, 128), jnp.float32)
    k = jax.random.normal(ks[1], (1, 8, seq, 128), jnp.float32)
    v = jax.random.normal(ks[2], (1, 8, seq, 128), jnp.float32)
    t_xla = bench(xla_fn, q, k, v)
    t_pallas = bench(pallas_fn, q, k, v)
    print(f"{seq:>8} | {t_xla*1e3:>10.3f} | {t_pallas*1e3:>12.3f} | {t_xla/t_pallas:>7.2f}x")


## 5. Tiny LLaMA decoder powered by the kernel

A small, randomly-initialized LLaMA (RMSNorm + RoPE + SwiGLU) whose causal
self-attention is our Pallas kernel. The model is untrained — the point is that
the **full inference path** flows through the custom kernel.

In [ ]:
import jax.numpy as jnp

DIM, N_LAYERS, N_HEADS, HEAD_DIM, FFN, VOCAB = 256, 4, 2, 128, 688, 256
EPS, THETA = 1e-5, 10000.0

def rms_norm(x, w):
    var = jnp.mean(jnp.square(x.astype(jnp.float32)), -1, keepdims=True)
    return (x.astype(jnp.float32) * jax.lax.rsqrt(var + EPS) * w).astype(x.dtype)

def rope_tables(seq, hd):
    half = hd // 2
    inv = 1.0 / (THETA ** (jnp.arange(0, half, dtype=jnp.float32) / half))
    f = jnp.outer(jnp.arange(seq, dtype=jnp.float32), inv)
    emb = jnp.concatenate([f, f], -1)
    return jnp.cos(emb), jnp.sin(emb)

def rotate_half(x):
    h = x.shape[-1] // 2
    return jnp.concatenate([-x[..., h:], x[..., :h]], -1)

def apply_rope(x, cos, sin):
    return x * cos[None, None] + rotate_half(x) * sin[None, None]

def init(key):
    ks = iter(jax.random.split(key, 4 + N_LAYERS * 7))
    nrm = lambda k, sh, s: jax.random.normal(k, sh, jnp.float32) * s
    ps = 1.0 / DIM ** 0.5
    p = {"embed": nrm(next(ks), (VOCAB, DIM), 0.02), "layers": []}
    for _ in range(N_LAYERS):
        p["layers"].append({
            "an": jnp.ones((DIM,)), "fn": jnp.ones((DIM,)),
            "wq": nrm(next(ks), (DIM, DIM), ps), "wk": nrm(next(ks), (DIM, DIM), ps),
            "wv": nrm(next(ks), (DIM, DIM), ps), "wo": nrm(next(ks), (DIM, DIM), ps),
            "wg": nrm(next(ks), (DIM, FFN), ps), "wu": nrm(next(ks), (DIM, FFN), ps),
            "wd": nrm(next(ks), (FFN, DIM), 1.0 / FFN ** 0.5)})
    p["final"] = jnp.ones((DIM,))
    p["head"] = nrm(next(ks), (DIM, VOCAB), ps)
    return p

def forward(p, tokens):
    b, seq = tokens.shape
    x = p["embed"][tokens]
    cos, sin = rope_tables(seq, HEAD_DIM)
    heads = lambda t: t.reshape(b, seq, N_HEADS, HEAD_DIM).transpose(0, 2, 1, 3)
    for L in p["layers"]:
        h = rms_norm(x, L["an"])
        q = apply_rope(heads(h @ L["wq"]), cos, sin)
        k = apply_rope(heads(h @ L["wk"]), cos, sin)
        v = heads(h @ L["wv"])
        a = flash_attention(q, k, v, causal=True)
        a = a.transpose(0, 2, 1, 3).reshape(b, seq, DIM)
        x = x + a @ L["wo"]
        h = rms_norm(x, L["fn"])
        x = x + (jax.nn.silu(h @ L["wg"]) * (h @ L["wu"])) @ L["wd"]
    return rms_norm(x, p["final"]) @ p["head"]

def generate(p, prompt, n):
    toks = prompt
    for _ in range(n):
        nxt = jnp.argmax(forward(p, toks)[:, -1, :], -1, keepdims=True)
        toks = jnp.concatenate([toks, nxt], 1)
    return toks

params = init(jax.random.PRNGKey(0))
prompt = jnp.array([[72, 101, 108, 108, 111]])  # "Hello" bytes
out = generate(params, prompt, 16)
print("Prompt:", prompt.tolist()[0])
print("Output:", out.tolist()[0])


## 6. Grouped-query attention (GQA / MQA)

Modern LLaMA models share each group of query heads over a single key/value head,
shrinking the KV cache. Our kernel supports this for free: `k`/`v` simply carry
fewer heads and the K/V `BlockSpec` maps query head `h` to KV head
`h // (num_heads // num_kv_heads)`. Here 8 query heads share 2 KV heads.

In [ ]:
def repeat_kv(x, n):
    return jnp.repeat(x, n, axis=1)

ks = jax.random.split(jax.random.PRNGKey(3), 3)
qg = jax.random.normal(ks[0], (1, 8, 256, 128), jnp.float32)   # 8 query heads
kg = jax.random.normal(ks[1], (1, 2, 256, 128), jnp.float32)   # 2 KV heads
vg = jax.random.normal(ks[2], (1, 2, 256, 128), jnp.float32)
out_gqa = flash_attention(qg, kg, vg, causal=True)
ref_gqa = reference_attention(qg, repeat_kv(kg, 4), repeat_kv(vg, 4), causal=True)
print("GQA (8 q-heads / 2 kv-heads) max abs error:",
      float(jnp.max(jnp.abs(out_gqa - ref_gqa))))


## 7. KV-cache decoding

Generating token-by-token by re-running the whole prefix is O(T²). A **KV cache**
stores each layer's keys/values so a decode step only computes the *new* token and
attends it against the cache. The cache here is **preallocated** to a fixed
capacity and written in place; `kv_len` tells the kernel how much of it is real,
and blocks past that point are skipped entirely. It produces the identical tokens
to the naive loop.

In [ ]:
def init_cache(b, cache_size, n_layers=N_LAYERS):
    shape = (b, N_HEADS, cache_size, HEAD_DIM)
    return [[jnp.zeros(shape), jnp.zeros(shape)] for _ in range(n_layers)]

def prefill(params, tokens, cache_size):
    b, seq = tokens.shape
    x = params["embed"][tokens]
    cos, sin = rope_tables(seq, HEAD_DIM)
    hd = lambda t, nh: t.reshape(b, seq, nh, HEAD_DIM).transpose(0, 2, 1, 3)
    cache = init_cache(b, cache_size)
    for li, L in enumerate(params["layers"]):
        h = rms_norm(x, L["an"])
        q = apply_rope(hd(h @ L["wq"], N_HEADS), cos, sin)
        k = apply_rope(hd(h @ L["wk"], N_HEADS), cos, sin)
        v = hd(h @ L["wv"], N_HEADS)
        a = flash_attention(q, k, v, causal=True).transpose(0, 2, 1, 3).reshape(b, seq, DIM)
        x = x + a @ L["wo"]
        # write the prompt's K/V into slots [0, seq) of the fixed-size cache
        cache[li] = [jax.lax.dynamic_update_slice_in_dim(cache[li][0], k, 0, 2),
                     jax.lax.dynamic_update_slice_in_dim(cache[li][1], v, 0, 2)]
        h = rms_norm(x, L["fn"])
        x = x + (jax.nn.silu(h @ L["wg"]) * (h @ L["wu"])) @ L["wd"]
    return (rms_norm(x, params["final"]) @ params["head"]), cache

def decode_step(params, token, cache, pos):
    b = token.shape[0]
    x = params["embed"][token]
    cos_all, sin_all = rope_tables(pos + 1, HEAD_DIM)
    cos, sin = cos_all[pos:pos + 1], sin_all[pos:pos + 1]
    hd = lambda t, nh: t.reshape(b, 1, nh, HEAD_DIM).transpose(0, 2, 1, 3)
    new_cache = []
    for L, (ck, cv) in zip(params["layers"], cache):
        h = rms_norm(x, L["an"])
        q = apply_rope(hd(h @ L["wq"], N_HEADS), cos, sin)
        k = apply_rope(hd(h @ L["wk"], N_HEADS), cos, sin)
        v = hd(h @ L["wv"], N_HEADS)
        # write this token's K/V at slot `pos`, then attend slots [0, pos]
        k = jax.lax.dynamic_update_slice_in_dim(ck, k, pos, 2)
        v = jax.lax.dynamic_update_slice_in_dim(cv, v, pos, 2)
        a = flash_attention(q, k, v, causal=False, kv_len=pos + 1, block_q=1)
        a = a.transpose(0, 2, 1, 3).reshape(b, 1, DIM)
        x = x + a @ L["wo"]
        new_cache.append([k, v])
        h = rms_norm(x, L["fn"])
        x = x + (jax.nn.silu(h @ L["wg"]) * (h @ L["wu"])) @ L["wd"]
    return (rms_norm(x, params["final"]) @ params["head"]), new_cache

def generate_cached(params, prompt, n):
    b, P = prompt.shape
    logits, cache = prefill(params, prompt, cache_size=P + n)
    nxt = jnp.argmax(logits[:, -1, :], -1, keepdims=True)
    toks = jnp.concatenate([prompt, nxt], 1)
    for i in range(n - 1):
        logits, cache = decode_step(params, nxt, cache, P + i)
        nxt = jnp.argmax(logits[:, -1, :], -1, keepdims=True)
        toks = jnp.concatenate([toks, nxt], 1)
    return toks

prompt = jnp.array([[72, 101, 108, 108, 111]])
naive = generate(params, prompt, 16)
cached = generate_cached(params, prompt, 16)
print("naive :", naive.tolist()[0])
print("cached:", cached.tolist()[0])
print("identical:", bool(jnp.array_equal(naive, cached)))


## 8. bfloat16

The kernel up-casts inputs to float32 *inside* VMEM, so the online softmax and
`p @ v` accumulate in float32 while HBM traffic halves. Outputs come back in the
input dtype — on a v3-8 this is usually the fastest setting for inference.

In [ ]:
qb, kb, vb = (x.astype(jnp.bfloat16) for x in (q, k, v))
out_bf16 = flash_attention(qb, kb, vb, causal=True)
ref_f32 = reference_attention(q, k, v, causal=True)
print("output dtype:", out_bf16.dtype)
print("max abs error vs f32 reference:",
      float(jnp.max(jnp.abs(out_bf16.astype(jnp.float32) - ref_f32))))


## 9. The backward pass (also Pallas)

The library version of this kernel is a `jax.custom_vjp` whose backward pass is
**two more Pallas kernels**. The forward saves the per-row log-sum-exp, so the
backward can recompute `p = exp(s - lse)` tile-by-tile:

```
delta_i = Σ_d o_id·do_id;   dv_j = Σ_i p_ij·do_i
ds_ij   = p_ij (do_i·v_j − delta_i)
dq_i    = scale·Σ_j ds_ij·k_j     (grid: b, h, q_block, kv_block)
dk_j    = scale·Σ_i ds_ij·q_i     (grid: b, h, kv_block, q_block)
```

dQ wants an outer loop over query blocks and dK/dV over kv blocks, hence two
kernels with transposed grids. Install the repo to use it with `jax.grad`:

```bash
!pip install -q git+https://github.com/aabhimittal/TPU-Native-Pallas-FlashAttention.git
```

```python
from pallas_flash import flash_attention
loss = lambda q, k, v: jnp.sum(flash_attention(q, k, v, causal=True))
dq, dk, dv = jax.grad(loss, argnums=(0, 1, 2))(q, k, v)
```

The cell below verifies the *math* of that backward against JAX autodiff using
the reference attention, so it runs standalone in this notebook.

In [ ]:
cot = jax.random.normal(jax.random.PRNGKey(11), q.shape, jnp.float32)

def manual_bwd(q, k, v, do, causal=True):
    """The FlashAttention backward math, written out (unfused, for checking)."""
    scale = 1.0 / q.shape[-1] ** 0.5
    s = jnp.einsum("bhqd,bhkd->bhqk", q, k) * scale
    if causal:
        qp = jax.lax.broadcasted_iota(jnp.int32, s.shape[-2:], 0)
        kp = jax.lax.broadcasted_iota(jnp.int32, s.shape[-2:], 1)
        s = jnp.where(qp >= kp, s, -1e30)
    lse = jax.scipy.special.logsumexp(s, axis=-1, keepdims=True)
    p = jnp.exp(s - lse)
    o = jnp.einsum("bhqk,bhkd->bhqd", p, v)
    delta = jnp.sum(o * do, axis=-1, keepdims=True)
    dv = jnp.einsum("bhqk,bhqd->bhkd", p, do)
    dp = jnp.einsum("bhqd,bhkd->bhqk", do, v)
    ds = p * (dp - delta)
    dq = jnp.einsum("bhqk,bhkd->bhqd", ds, k) * scale
    dk = jnp.einsum("bhqk,bhqd->bhkd", ds, q) * scale
    return dq, dk, dv

autodiff = jax.grad(lambda q, k, v: jnp.sum(reference_attention(q, k, v, causal=True) * cot),
                    argnums=(0, 1, 2))(q, k, v)
manual = manual_bwd(q, k, v, cot)
for name, a, b in zip(("dq", "dk", "dv"), manual, autodiff):
    rel = float(jnp.max(jnp.abs(a - b)) / (jnp.max(jnp.abs(b)) + 1e-9))
    print(f"{name}: relative error vs autodiff = {rel:.2e}")


## 10. Fused RoPE (rotate inside the kernel)

Applying RoPE the ordinary way costs two extra full round-trips of Q and K
through HBM — read, rotate, write, read back — for zero extra math. The kernel
already streams Q and K into VMEM, so it can rotate them *there* and never write
the rotated copies out at all.

The cos/sin tables become ordinary kernel inputs with their own `BlockSpec`s
(query tables follow the q block, key tables the kv block), so each grid step
pulls in exactly the rows it needs.

In [ ]:
def rope_tables_2d(seq, hd, theta=10000.0):
    half = hd // 2
    inv = 1.0 / (theta ** (jnp.arange(0, half, dtype=jnp.float32) / half))
    f = jnp.outer(jnp.arange(seq, dtype=jnp.float32), inv)
    emb = jnp.concatenate([f, f], -1)
    return jnp.cos(emb), jnp.sin(emb)          # [seq, head_dim]

def _rope_kernel(q_ref, k_ref, v_ref, qc_ref, qs_ref, kc_ref, ksn_ref, o_ref,
                 m_scratch, l_scratch, acc_scratch,
                 *, sm_scale, causal, block_q, block_k, kv_len):
    i, j = pl.program_id(2), pl.program_id(3)
    nkv = pl.num_programs(3)

    @pl.when(j == 0)
    def _init():
        m_scratch[...] = jnp.full_like(m_scratch, _NEG_INF)
        l_scratch[...] = jnp.zeros_like(l_scratch)
        acc_scratch[...] = jnp.zeros_like(acc_scratch)

    def rot_half(x):
        h = x.shape[-1] // 2
        return jnp.concatenate([-x[..., h:], x[..., :h]], -1)

    def _do():
        q = q_ref[0, 0].astype(jnp.float32)
        k = k_ref[0, 0].astype(jnp.float32)
        v = v_ref[0, 0].astype(jnp.float32)
        # --- the fusion: rotate while the blocks are already in VMEM ---
        q = q * qc_ref[...] + rot_half(q) * qs_ref[...]
        k = k * kc_ref[...] + rot_half(k) * ksn_ref[...]

        valid = j * block_k + jax.lax.broadcasted_iota(jnp.int32, (block_k, 1), 0) < kv_len
        k = jnp.where(valid, k, 0.0); v = jnp.where(valid, v, 0.0)
        s = jnp.dot(q, k.T, preferred_element_type=jnp.float32) * sm_scale
        qp = i * block_q + jax.lax.broadcasted_iota(jnp.int32, (block_q, block_k), 0)
        kp = j * block_k + jax.lax.broadcasted_iota(jnp.int32, (block_q, block_k), 1)
        mask = kp < kv_len
        if causal:
            mask = jnp.logical_and(mask, qp >= kp)
        s = jnp.where(mask, s, _NEG_INF)

        m_prev = m_scratch[...]
        m_new = jnp.maximum(m_prev, jnp.max(s, -1, keepdims=True))
        p = jnp.exp(s - m_new); alpha = jnp.exp(m_prev - m_new)
        l_scratch[...] = alpha * l_scratch[...] + jnp.sum(p, -1, keepdims=True)
        acc_scratch[...] = acc_scratch[...] * alpha + jnp.dot(p, v, preferred_element_type=jnp.float32)
        m_scratch[...] = m_new

    live = j * block_k < kv_len
    if causal:
        live = jnp.logical_and(live, i * block_q + (block_q - 1) >= j * block_k)
    pl.when(live)(_do)

    @pl.when(j == nkv - 1)
    def _fin():
        l = jnp.where(l_scratch[...] == 0.0, 1.0, l_scratch[...])
        o_ref[0, 0] = (acc_scratch[...] / l).astype(o_ref.dtype)


def flash_attention_rope(q, k, v, qc, qs, kc=None, ks_=None, *, causal=False,
                         block_q=128, block_k=128, kv_len=None, interpret=False):
    if kc is None: kc, ks_ = qc, qs
    b, H, Sq, D = q.shape
    Hkv, Sk = k.shape[1], k.shape[2]
    qpk = H // Hkv
    if kv_len is None: kv_len = Sk
    grid = (b, H, pl.cdiv(Sq, block_q), pl.cdiv(Sk, block_k))
    qsp = pl.BlockSpec((1, 1, block_q, D), lambda b, h, i, j: (b, h, i, 0))
    ksp = pl.BlockSpec((1, 1, block_k, D), lambda b, h, i, j: (b, h // qpk, j, 0))
    qtb = pl.BlockSpec((block_q, D), lambda b, h, i, j: (i, 0))   # q rope table
    ktb = pl.BlockSpec((block_k, D), lambda b, h, i, j: (j, 0))   # k rope table
    return pl.pallas_call(
        functools.partial(_rope_kernel, sm_scale=1.0 / D ** 0.5, causal=causal,
                          block_q=block_q, block_k=block_k, kv_len=kv_len),
        grid=grid, in_specs=[qsp, ksp, ksp, qtb, qtb, ktb, ktb], out_specs=qsp,
        out_shape=jax.ShapeDtypeStruct(q.shape, q.dtype),
        scratch_shapes=[pltpu.VMEM((block_q, 1), jnp.float32),
                        pltpu.VMEM((block_q, 1), jnp.float32),
                        pltpu.VMEM((block_q, D), jnp.float32)],
        compiler_params=pltpu.CompilerParams(
            dimension_semantics=("parallel", "parallel", "parallel", "arbitrary")),
        interpret=interpret, name="flash_attention_rope")(q, k, v, qc, qs, kc, ks_)


S = q.shape[2]
cos2, sin2 = rope_tables_2d(S, 128)
def apply_rope_xla(x, cos, sin):
    h = x.shape[-1] // 2
    rot = jnp.concatenate([-x[..., h:], x[..., :h]], -1)
    return x * cos[None, None] + rot * sin[None, None]

fused = flash_attention_rope(q, k, v, cos2, sin2, causal=True)
unfused = flash_attention(apply_rope_xla(q, cos2, sin2), apply_rope_xla(k, cos2, sin2), v, causal=True)
print("fused vs unfused RoPE, max abs error:", float(jnp.max(jnp.abs(fused - unfused))))


## 11. Paged attention (batched serving)

A contiguous cache forces you to preallocate `max_seq_len` per sequence, so a
batch of mostly-short sequences wastes most of what it reserves. **Paged
attention** keeps one pool of fixed-size pages plus a per-sequence **block
table** mapping logical → physical pages.

The TPU-interesting part: a Pallas `index_map` must be a pure function of the
grid indices and *cannot* read a normal input array. The block table therefore
lives in **scalar-prefetch** memory (`pltpu.PrefetchScalarGridSpec`), which index
maps can read — so the DMA that streams a page into VMEM is addressed by a
runtime table lookup.

In [ ]:
def _paged_kernel(bt_ref, cl_ref, q_ref, k_ref, v_ref, o_ref,
                  m_scratch, l_scratch, acc_scratch, *, sm_scale, page_size):
    b, p = pl.program_id(0), pl.program_id(2)
    npages = pl.num_programs(2)
    ctx = cl_ref[b]

    @pl.when(p == 0)
    def _init():
        m_scratch[...] = jnp.full_like(m_scratch, _NEG_INF)
        l_scratch[...] = jnp.zeros_like(l_scratch)
        acc_scratch[...] = jnp.zeros_like(acc_scratch)

    def _do():
        qq = q_ref[0, 0].astype(jnp.float32)
        kk = k_ref[0, 0].astype(jnp.float32)
        vv = v_ref[0, 0].astype(jnp.float32)
        pos = p * page_size + jax.lax.broadcasted_iota(jnp.int32, (page_size, 1), 0)
        valid = pos < ctx
        kk = jnp.where(valid, kk, 0.0); vv = jnp.where(valid, vv, 0.0)
        s = jnp.dot(qq, kk.T, preferred_element_type=jnp.float32) * sm_scale
        s = jnp.where(valid.reshape(1, page_size), s, _NEG_INF)
        m_prev = m_scratch[...]
        m_new = jnp.maximum(m_prev, jnp.max(s, -1, keepdims=True))
        pr = jnp.exp(s - m_new); alpha = jnp.exp(m_prev - m_new)
        l_scratch[...] = alpha * l_scratch[...] + jnp.sum(pr, -1, keepdims=True)
        acc_scratch[...] = acc_scratch[...] * alpha + jnp.dot(pr, vv, preferred_element_type=jnp.float32)
        m_scratch[...] = m_new

    pl.when(p * page_size < ctx)(_do)   # skip pages past this sequence

    @pl.when(p == npages - 1)
    def _fin():
        l = jnp.where(l_scratch[...] == 0.0, 1.0, l_scratch[...])
        o_ref[0, 0] = (acc_scratch[...] / l).astype(o_ref.dtype)


def paged_attention(q, k_pages, v_pages, block_tables, context_lens, interpret=False):
    b, H, _, D = q.shape
    npages, Hkv, page_size, _ = k_pages.shape
    qpk, pps = H // Hkv, block_tables.shape[1]

    # index maps receive the grid indices FOLLOWED BY the scalar-prefetch refs
    def kmap(bi, h, p, bt, cl): return (bt[bi, p], h // qpk, 0, 0)
    def qmap(bi, h, p, bt, cl): return (bi, h, 0, 0)

    gs = pltpu.PrefetchScalarGridSpec(
        num_scalar_prefetch=2, grid=(b, H, pps),
        in_specs=[pl.BlockSpec((1, 1, 1, D), qmap),
                  pl.BlockSpec((1, 1, page_size, D), kmap),
                  pl.BlockSpec((1, 1, page_size, D), kmap)],
        out_specs=pl.BlockSpec((1, 1, 1, D), qmap),
        scratch_shapes=[pltpu.VMEM((1, 1), jnp.float32),
                        pltpu.VMEM((1, 1), jnp.float32),
                        pltpu.VMEM((1, D), jnp.float32)])
    return pl.pallas_call(
        functools.partial(_paged_kernel, sm_scale=1.0 / D ** 0.5, page_size=page_size),
        grid_spec=gs, out_shape=jax.ShapeDtypeStruct(q.shape, q.dtype),
        compiler_params=pltpu.CompilerParams(
            dimension_semantics=("arbitrary", "arbitrary", "arbitrary")),
        interpret=interpret, name="paged_attention")(
            block_tables.astype(jnp.int32), context_lens.astype(jnp.int32), q, k_pages, v_pages)


# A batch of 3 sequences with very different lengths, pages deliberately scrambled.
import numpy as np
PAGE, NPAGES, D = 64, 16, 128
lens = [10, 130, 200]
pps = max(lens) // PAGE + 1
kk = jax.random.split(jax.random.PRNGKey(21), 3)
qd = jax.random.normal(kk[0], (3, 4, 1, D), jnp.float32)
kp = jax.random.normal(kk[1], (NPAGES, 2, PAGE, D), jnp.float32)
vp = jax.random.normal(kk[2], (NPAGES, 2, PAGE, D), jnp.float32)
rng = np.random.default_rng(0)
bt = jnp.asarray(np.stack([rng.choice(NPAGES, pps, replace=False) for _ in range(3)]), jnp.int32)
cl = jnp.asarray(lens, jnp.int32)

out_paged = paged_attention(qd, kp, vp, bt, cl)
QPK = qd.shape[1] // kp.shape[1]          # query heads per kv head (GQA)
for b_i, L in enumerate(lens):
    kc = jnp.concatenate([kp[int(bt[b_i, p])] for p in range(pps)], axis=1)[:, :L]
    vc = jnp.concatenate([vp[int(bt[b_i, p])] for p in range(pps)], axis=1)[:, :L]
    # the section-3 reference is plain MHA, so expand the kv heads by hand
    ref = reference_attention(qd[b_i:b_i+1],
                              jnp.repeat(kc[None], QPK, axis=1),
                              jnp.repeat(vc[None], QPK, axis=1), causal=False)
    print(f"seq {b_i} (len {L:3d}): max abs error vs contiguous = "
          f"{float(jnp.max(jnp.abs(out_paged[b_i:b_i+1] - ref))):.2e}")


## 12. Ring-buffer cache (unbounded streaming)

A fixed cache still needs capacity == total length. A **ring buffer** of capacity
`C` writes position `pos` into slot `pos % C`, evicting the token from `C` steps
ago — so generation runs forever in constant memory (sliding-window attention).

No extra masking is needed: RoPE is applied *before* caching so each entry
carries its absolute position, and attention is permutation-invariant over keys,
so the ring's out-of-order layout does not matter. The cell below demonstrates
exactly that invariance.

In [ ]:
C = 128
kk = jax.random.split(jax.random.PRNGKey(31), 3)
q1 = jax.random.normal(kk[0], (1, 2, 1, 128), jnp.float32)
kr = jax.random.normal(kk[1], (1, 2, C, 128), jnp.float32)
vr = jax.random.normal(kk[2], (1, 2, C, 128), jnp.float32)

# A wrapped ring is just a rotation of the key/value axis.
in_order = flash_attention(q1, kr, vr, causal=False, block_q=1, block_k=C)
rolled = flash_attention(q1, jnp.roll(kr, 37, axis=2), jnp.roll(vr, 37, axis=2),
                         causal=False, block_q=1, block_k=C)
print("ring rotation changes nothing:", float(jnp.max(jnp.abs(in_order - rolled))))

# Partially-filled ring: kv_len = min(total_written, C)
filled = 20
partial = flash_attention(q1, kr, vr, causal=False, block_q=1, block_k=C, kv_len=filled)
prefix = reference_attention(q1, kr[:, :, :filled], vr[:, :, :filled], causal=False)
print("partial ring vs true prefix:", float(jnp.max(jnp.abs(partial - prefix))))


## 13. Notes & extensions

- **Why it's faster:** the kernel keeps the `[block_q, block_k]` score tile in
  VMEM and streams K/V blocks, so HBM traffic and peak memory grow ~linearly in
  sequence length instead of quadratically — the gap widens as `seq_len` grows.
- **Manual DMA (advanced):** `BlockSpec` lets Pallas pipeline the HBM→VMEM
  copies for you. For fully manual control you can instead keep K/V in
  `memory_space=pltpu.ANY` and issue `pltpu.make_async_copy(src, dst, sem)`
  yourself inside the kernel — the same online-softmax math applies.
- **KV cache:** implemented above on a fixed-capacity buffer with `kv_len`
  masking + block skipping, so decode cost tracks the filled length. A
  ring-buffer cache would extend this to unbounded streaming.
- **Backward pass:** the library ships Pallas dQ and dK/dV kernels wired up via
  `jax.custom_vjp` (section 9), so `jax.grad` works end to end.
- **Next:** fused RoPE inside the kernel, and paged attention for batched serving.